# 01 — Getting started with laser-init

This notebook walks through the simplest end-to-end `laser-init` workflow:
generate a dataset for a country, inspect the outputs, and run the model.

**Prerequisites:** `laser-init` installed and on your `PATH`, and a network
connection (the first run downloads data into the cache).

We use **ETH** (admin level 2) here; change `COUNTRY` below to try
another. Larger countries take longer to model.

In [ ]:
import subprocess
from pathlib import Path

COUNTRY, LEVEL, START, END = "ETH", 2, 2015, 2017
BASE = Path(COUNTRY) / str(START)

## Generate the dataset

`laser-init <COUNTRY> <LEVEL> <START_YEAR> <END_YEAR>` downloads boundaries,
population, and demographics, then writes a ready-to-run model into
`./<COUNTRY>/<START_YEAR>`.

In [ ]:
if not (BASE / "config.yaml").exists():
    subprocess.run(["laser-init", COUNTRY, str(LEVEL), str(START), str(END)], check=True)

sorted(p.name for p in BASE.iterdir())

## Inspect the GeoPackage

The `.gpkg` is the core spatial product: one row per administrative unit with a
`population` and a `geometry`.

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(BASE / f"{COUNTRY}_admin{LEVEL}.gpkg")
print(f"units: {len(gdf)}, total population: {gdf.population.sum():,.0f}")
gdf[["name", "population"]].head()

## Built-in validation plots

`laser-init` also writes validation figures. Here is the population choropleth:

In [ ]:
from IPython.display import Image

Image(filename=str(BASE / "choropleth.png"))

## Run the model

The generated `seir.py` runs the simulation and writes `seir_output.pdf`.

**Note:** this runs a full LASER simulation and can take a while for large
countries.

In [ ]:
subprocess.run(["python3", "./seir.py"], cwd=BASE, check=True)
print("done:", (BASE / "seir_output.pdf").exists())

## Next steps

- `02_data_exploration.ipynb` — explore the output data in depth
- `03_model_comparison.ipynb` — compare SI / SIR / SEIR dynamics
- `04_custom_analysis.ipynb` — an end-to-end custom analysis